In [1]:
!pip install scikit-learn==1.2.0
!pip install pandas as pd
!pip install joblib

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 35.1 MB/s eta 0:00:00a 0:00:01
ERROR: Ignored the following versions that require a different python version: 3.0.0 Requires-Python >=3.11; 3.0.0rc0 Requires-Python >=3.11; 3.0.0rc1 Requires-Python >=3.11; 3.0.0rc2 Requires-Python >=3.11; 3.0.1 Requires-Python >=3.11; 3.0.2 Requires-Python >=3.11; 3.0.3 Requires-Python >=3.11
ERROR: Could not find a version that satisfies the requirement as (from versions: none)
ERROR: No matching distribution found for as

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --

In [2]:
import pandas as pd 
import numpy as np
import joblib


ModuleNotFoundError: No module named 'pandas'

In [ ]:
model = joblib.load("/Users/sadiqkhawaja/Desktop/PhD/CVD_RiskStrat_WebApp-merge-interface2/model_files/batch_models/TAVIModel.pkl")

In [ ]:
exported_forest = []

for tree in model.estimators_:
    exported_forest.append({
        "feature": tree.tree_.feature.copy(),
        "threshold": tree.tree_.threshold.copy(),
        "left": tree.tree_.children_left.copy(),
        "right": tree.tree_.children_right.copy(),
        "value": tree.tree_.value[:,0,:].copy()
    })

In [ ]:
forest_export = {
    "n_features_in": model.n_features_in_,
    "classes": model.classes_.tolist(),
    "trees": exported_forest
}

In [ ]:
backendValues = {
    "Age":94,
    "Diabetes":0,
    "Ex_smoking":0,
    "Hypertension":1,
    "Creatinine":68,
    "Pre-operative_heart_rhythm":0,
    "Baseline_Hb":125,
    "Poor_mobility":1,
    "FEV1":1.6,
    "Predicted_VC":157,
    "FEV1_VC":0.727273,
    "Katz_Index":5,
}

In [ ]:
feature_order = [
    "Age",
    "Diabetes",
    "Ex_smoking",
    "Hypertension",
    "Creatinine",
    "Pre-operative_heart_rhythm",
    "Baseline_Hb",
    "Poor_mobility",
    "FEV1",
    "Predicted_VC",
    "FEV1_VC",
    "Katz_Index",
]

x = np.array([backendValues[f] for f in feature_order])

In [ ]:
def predict_tree(tree, x):
    node = 0

    while tree["left"][node] != tree["right"][node]:
        feat = tree["feature"][node]
        thresh = tree["threshold"][node]

        if x[feat] <= thresh:
            node = tree["left"][node]
        else:
            node = tree["right"][node]

    return tree["value"][node]

In [ ]:
def predict_forest_sklearn_style(forest_export, x):

    n_classes = len(forest_export["classes"])
    total = np.zeros(n_classes)

    for tree in forest_export["trees"]:

        leaf = predict_tree(tree, x).astype(float)

        # Convert this tree's leaf counts to probabilities
        tree_prob = leaf / leaf.sum()

        total += tree_prob

    return total / len(forest_export["trees"])

In [ ]:
sk_prob = model.predict_proba([x])[0]
manual_prob = predict_forest_sklearn_style(forest_export, x)

print("sklearn :", sk_prob)
print("manual  :", manual_prob)
print("max diff:", np.max(np.abs(sk_prob - manual_prob)))
print("sklearn :", sk_prob)
print("manual  :", manual_prob)

print("max diff:", np.max(np.abs(sk_prob - manual_prob)))

In [ ]:
print("Trees exported:", len(exported_forest))
print("Model trees:", len(model.estimators_))

In [ ]:
dir(model)

In [ ]:
len(model.feature_importances_)